<a href="https://colab.research.google.com/github/jashwanthd19/IDRA-Assignments/blob/main/Used_Car_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer


df = pd.read_csv("Day12_Used_Car_Preprocessing_Dataset.csv")


target = "Resale_Price_Lakh"

X = df.drop(columns=[target, "Car_ID"])
y = df[target]


numeric_cols = [
    "Year",
    "Mileage_Km",
    "Engine_CC",
    "Power_BHP",
    "Previous_Owners",
    "Accidents_Reported",
    "Service_Score"
]

nominal_cols = [
    "Brand",
    "Fuel_Type",
    "Transmission",
    "City",
    "Seller_Type"
]

ordinal_cols = [
    "Condition"
]


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


outlier_cols = [
    "Mileage_Km",
    "Engine_CC",
    "Power_BHP"
]

X_train = X_train.copy()
X_test = X_test.copy()

for col in outlier_cols:

    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    X_train[col] = X_train[col].clip(lower, upper)
    X_test[col] = X_test[col].clip(lower, upper)


numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)


nominal_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ]
)


ordinal_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(
            categories=[
                ["Poor", "Fair", "Good", "Very Good", "Excellent"]
            ]
        ))
    ]
)


preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_cols),
        ("nominal", nominal_transformer, nominal_cols),
        ("ordinal", ordinal_transformer, ordinal_cols)
    ]
)


X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)


feature_names = preprocessor.get_feature_names_out()


X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=feature_names
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=feature_names
)


X_train_processed[target] = y_train.reset_index(drop=True)
X_test_processed[target] = y_test.reset_index(drop=True)


print("Training Data Shape:", X_train_processed.shape)
print("Testing Data Shape:", X_test_processed.shape)

print("\nMissing Values in Training Data:")
print(X_train_processed.isnull().sum().sum())

print("\nMissing Values in Testing Data:")
print(X_test_processed.isnull().sum().sum())

print("\nProcessed Training Data:")
print(X_train_processed.head())

print("\nProcessed Testing Data:")
print(X_test_processed.head())


X_train_processed.to_csv(
    "used_car_preprocessed_train.csv",
    index=False
)

X_test_processed.to_csv(
    "used_car_preprocessed_test.csv",
    index=False
)

Training Data Shape: (256, 38)
Testing Data Shape: (64, 38)

Missing Values in Training Data:
0

Missing Values in Testing Data:
0

Processed Training Data:
   numeric__Year  numeric__Mileage_Km  numeric__Engine_CC  numeric__Power_BHP  \
0      -0.486391            -0.225883           -0.322882            0.304485   
1       0.725445             0.089371           -0.656431           -0.330444   
2      -1.395268             1.115798           -0.120529            0.420784   
3       1.028404            -0.195076            0.479859            0.414498   
4      -1.092309             0.706680            0.786724           -0.437314   

   numeric__Previous_Owners  numeric__Accidents_Reported  \
0                 -0.734379                    -0.442634   
1                 -0.734379                     1.477948   
2                 -0.734379                    -0.442634   
3                  0.433329                    -0.442634   
4                  0.433329                    -0.442634